# Notebook 04 — ADCD-Net setup and inference (Week 5)

**Goal of this notebook:** get the pretrained ADCD-Net detector running on Kaggle, feed it my TextCtrl forgeries, and save one probability map per page. Notebook 05 scores those maps.

| Section | Step | ⏱ Time | Needs GPU? |
|---|---|---|---|
| 1. CONFIG + run folder | P1 | < 5 s | no |
| 2. GPU check | P1 | < 5 s | yes |
| 3. Find the datasets + folder layout | P1 | < 10 s | no |
| 4. Look at results.csv and plan.csv | P1 | < 5 s | no |
| 5. Image sizes + file-name match | P1 | < 10 s | no |
| 6. Build the Python 3.10 venv | P2 | 1–2 min | no |
| 7. Install PyTorch 2.3.0+cu121 | P2 | 3–6 min | no |
| 8. Install the other packages, one at a time | P2 | 1–3 min | no |
| 9. Build jpeg2dct (the risky one) | P2 | 1–3 min | no |
| 10. Clone the ADCD-Net repo | P2 | < 15 s | no |
| 11. Final check: imports, versions, weights | P2 | 20–60 s | yes |
| *P3–P5 sections get added later (data adapter, OCR masks, smoke test, sanity check, full inference)* | | | |

**Total for P1–P2: about 10–15 minutes**, almost all of it waiting for downloads (sections 7–9). You can start your lit review once section 6 is running and just check back.

## 1. CONFIG and run folder (P1)

**WHY:** every setting lives in one place (seed, threshold, sizes), so anyone can re-run this and get the same numbers. Rule: set the seeds everywhere and save the settings as `config.json` inside the run folder. Each run gets its own dated folder, so an old run is never overwritten.

To **continue** an earlier run (e.g. after a session restart) instead of starting a new folder, set `RUN_NAME = "adcd_YYYYMMDD_HHMM"` to that folder's name.

⏱ < 5 s. ✅ You should see the run folder path (e.g. `/kaggle/working/adcd_runs/adcd_20261026_1700`) and the settings printed.

In [ ]:
# Tools from the standard library: files, json, randomness, timing, shell commands
import os, json, random, time, glob, subprocess
# Tool to read today's date and time (for the run-folder name)
from datetime import datetime
# numpy = maths on arrays, pandas = tables
import numpy as np
import pandas as pd
# This is KAGGLE's own torch, used here only for the GPU check and seeds (ADCD-Net itself runs in the venv)
import torch

# All settings in one dictionary, so they can be saved and reused
CONFIG = {
    # Seed for random, numpy and torch (same seed = same random choices every run)
    "seed": 42,
    # Probability cut-off: a pixel with probability >= 0.5 counts as "tampered"
    "threshold": 0.5,
    # ADCD-Net shrinks the longest side of each page to this many pixels
    "val_max_size": 512,
    # Number of bootstrap resamples for confidence intervals (used in notebook 05)
    "N_BOOT": 1000,
    # Dataset sizes for the size-trend analysis: 10, 20, ..., 100 (used in notebook 05)
    "SIZES": list(range(10, 101, 10)),
    # Number of random orderings for the size-trend curve (used in notebook 05)
    "N_ORDERINGS": 20,
    # The three input variants: tampered PNG, tampered JPEG q90, untouched originals
    "variants": ["png", "jpg90", "orig"],
    # Where the ADCD-Net code lives (repo is called ACDC-Net, paper is ADCD-Net)
    "repo_url": "https://github.com/KAHIMWONG/ACDC-Net",
    # Versions from the ADCD-Net README
    "python_version": "3.10",
    "torch_version": "2.3.0+cu121",
    "albumentations_version": "2.0.8",
    # Kaggle dataset names to search for (both spellings of the weights dataset)
    "dataset_slugs": {"run": ["PATH_TO_TEXTCTRL_RUN_DATASET"], "weights": ["PATH_TO_ADCD_WEIGHTS_DATASET", "adcd-weight"]},
}

# Leave as None for a new run; set to an old folder name to continue that run
RUN_NAME = None

# Seed Python's random module
random.seed(CONFIG["seed"])
# Seed numpy's random numbers
np.random.seed(CONFIG["seed"])
# Seed torch on CPU and on every GPU
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])

# Kaggle's writable folder (everything we save goes under here)
WORK = "/kaggle/working"
# Where the ADCD-Net repo will be cloned
REPO_DIR = f"{WORK}/ADCD-Net-code"
# Where our own helper scripts (run with the venv Python) are written
SCRIPTS_DIR = f"{WORK}/adcd_scripts"
# The Python 3.10 venv lives in /tmp so its ~5 GB is NOT saved as notebook output
VENV_DIR = "/tmp/venv310"
# The venv's Python interpreter (we call ADCD-Net scripts with this one)
VENV_PY = f"{VENV_DIR}/bin/python"

# If no old run was given, make a new name from today's date and time
if RUN_NAME is None:
    RUN_NAME = "adcd_" + datetime.now().strftime("%Y%m%d_%H%M")
# Full path of this run's folder
RUN_DIR = f"{WORK}/adcd_runs/{RUN_NAME}"
# Create the run folder, its figures folder and the scripts folder (no error if they exist)
os.makedirs(f"{RUN_DIR}/figures", exist_ok=True)
os.makedirs(SCRIPTS_DIR, exist_ok=True)

# Store the folder paths inside CONFIG too, so config.json records them
CONFIG["paths"] = {"run_dir": RUN_DIR, "repo_dir": REPO_DIR, "venv_py": VENV_PY}
# Save CONFIG as config.json in the run folder
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

# Show what we did
print("Run folder :", RUN_DIR)
print("Settings   :", {k: CONFIG[k] for k in ["seed", "threshold", "val_max_size", "N_BOOT", "N_ORDERINGS"]})
print("SIZES      :", CONFIG["SIZES"])
print("Saved      :", f"{RUN_DIR}/config.json")

## 2. GPU check (P1)

**WHY:** ADCD-Net is a big neural network; on a GPU one page takes well under a second, on CPU much longer. Before installing anything, confirm Kaggle actually gave us the GPUs.

⏱ < 5 s. ✅ Two lines like `Tesla T4, 15360 MiB`, then `CUDA available: True, GPUs: 2`. If you see "No GPU", open the right-hand panel → Session options → Accelerator → GPU T4 x2.

In [ ]:
# Ask the NVIDIA driver for each GPU's name and memory
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
# Print the answer, or a warning if the command found nothing
print(gpu.stdout.strip() if gpu.returncode == 0 else "No GPU found: set Accelerator = GPU T4 x2")
# Double-check from inside Python (Kaggle's torch)
print("CUDA available:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())

## 3. Find the two datasets and show the folder layout (P1)

**WHY:** Kaggle mounts uploaded datasets read-only, but the exact path depends on how they were uploaded (`/kaggle/input/datasets/PATH_TO_KAGGLE_USERNAME/<name>/` or `/kaggle/input/<name>/`), and zipped folders sometimes end up nested (a folder inside a folder). Instead of guessing, we search, print what we found, and count the files. If anything is nested or named differently from CLAUDE.md, the cell prints a ⚠️ line.

⏱ < 10 s. ✅ Two dataset paths; `ADCDNet.pth` and `docres.pkl` found; **100 files** each in `tampered`, `tampered_jpg`, `originals`, `masks/box`, `masks/diff`; plus `config.json`, `plan.csv`, `results.csv`.

In [ ]:
# Function: find a dataset folder by trying both Kaggle path styles for each name
def find_dataset(slugs):
    # Try every spelling of the name we were given
    for slug in slugs:
        # The two places Kaggle may mount it
        for candidate in [f"/kaggle/input/datasets/PATH_TO_KAGGLE_USERNAME/{slug}", f"/kaggle/input/{slug}"]:
            # Return the first one that exists
            if os.path.isdir(candidate):
                return candidate
    # Nothing found: show what IS mounted (max 20 names) to help fix the name
    print("⚠️ Not found:", slugs, "| mounted:", sorted(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/datasets/*/*"))[:20])
    return None

# Function: find the first file with this name at most 3 folder levels below `root`
def find_file(root, name):
    # Look at depth 0, 1, 2 and 3
    for pattern in [name, f"*/{name}", f"*/*/{name}", f"*/*/*/{name}"]:
        # glob returns every matching path
        hits = sorted(glob.glob(os.path.join(root, pattern)))
        # Return the first match
        if hits:
            return hits[0]
    # No match
    return None

# Locate the TextCtrl run dataset and the weights dataset
RUN_DS = find_dataset(CONFIG["dataset_slugs"]["run"])
W_DS = find_dataset(CONFIG["dataset_slugs"]["weights"])
print("TextCtrl dataset:", RUN_DS)
print("Weights dataset :", W_DS)

# The real run folder is wherever results.csv lives (it may be one level down, e.g. run_..._evaluated/)
SRC_RUN = os.path.dirname(find_file(RUN_DS, "results.csv"))
# Tell me if it is nested
if SRC_RUN != RUN_DS:
    print("⚠️ Run folder is nested, using the inner folder:", SRC_RUN)
# Find the two weight files wherever they are inside the weights dataset
CKPT_PATH = find_file(W_DS, "ADCDNet.pth")
DOCRES_PATH = find_file(W_DS, "docres.pkl")
print("ADCDNet.pth     :", CKPT_PATH)
print("docres.pkl      :", DOCRES_PATH)
# Tell me if the weights are not directly in the dataset (CLAUDE.md expects them at the top)
for p in [CKPT_PATH, DOCRES_PATH]:
    if p is not None and os.path.dirname(p) != W_DS:
        print("⚠️ Weight file is in a sub-folder:", p)

# Top-level files and folders of the run (max 20 names)
print("\nRun folder contents:", sorted(os.listdir(SRC_RUN))[:20])

# The sub-folders CLAUDE.md says should exist, and what they should hold
expected = ["tampered", "tampered_jpg", "originals", "masks/box", "masks/diff", "figures"]
# One row per sub-folder for a small summary table
rows = []
for sub in expected:
    # Full path of this sub-folder
    folder = os.path.join(SRC_RUN, sub)
    # If it is missing, record that and move on
    if not os.path.isdir(folder):
        rows.append({"folder": sub, "n_files": "MISSING", "extensions": "", "first 3 names": ""})
        continue
    # File names, sorted
    names = sorted(os.listdir(folder))
    # Count each file extension (e.g. .png: 100)
    exts = pd.Series([os.path.splitext(n)[1] for n in names]).value_counts().to_dict()
    # Add a row: folder, count, extensions, first 3 names
    rows.append({"folder": sub, "n_files": len(names), "extensions": exts, "first 3 names": names[:3]})
# Show the table
layout = pd.DataFrame(rows)
print(layout.to_string(index=False))

# Any other folders in the run (e.g. the 4-panel pictures folder): name and file count, max 20
known = {"tampered", "tampered_jpg", "originals", "masks", "figures"}
for extra in sorted(os.listdir(SRC_RUN))[:20]:
    # Only folders we did not already check above
    if os.path.isdir(os.path.join(SRC_RUN, extra)) and extra not in known:
        print(f"Extra folder (not used as ADCD input): {extra}/ -> {len(os.listdir(os.path.join(SRC_RUN, extra)))} files")

# Remember the paths in CONFIG and re-save config.json
CONFIG["paths"].update({"src_run": SRC_RUN, "ckpt": CKPT_PATH, "docres": DOCRES_PATH})
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

## 4. Look at results.csv and plan.csv (P1)

**WHY:** `results.csv` holds one row per forgery with everything we measured in Week 4 (source/target text, number vs word, OCR correctness, SSIM, ...). These columns are what we will later use to explain *why* ADCD-Net finds some forgeries and misses others. `plan.csv` holds the page path, which contains the bank name. Here we only check the columns and 3 rows; no analysis yet.

⏱ < 5 s. ✅ About 100 rows each; the columns listed in CLAUDE.md (`kind`, `has_upper`, `text_correct`, `ssim`, ...); the page path in `plan.csv` should show a folder that looks like a bank name.

In [ ]:
# Show all columns when printing tables, and cut long text at 60 characters
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)

# Read the two tables
results = pd.read_csv(os.path.join(SRC_RUN, "results.csv"))
plan = pd.read_csv(os.path.join(SRC_RUN, "plan.csv"))

# For each table: size, column names, first 3 rows
for name, table in [("results.csv", results), ("plan.csv", plan)]:
    print(f"===== {name}: {table.shape[0]} rows x {table.shape[1]} columns =====")
    print("Columns:", list(table.columns))
    display(table.head(3))

# Quick count of numbers vs words (Week 4 had 33 numbers and 67 words)
if "kind" in results.columns:
    print("kind counts:", results["kind"].value_counts().to_dict())

## 5. Image sizes and file-name match (P1)

**WHY:** ADCD-Net shrinks each page so its longest side is 512 px. Our pages should be about 2480 px wide, i.e. shrunk about 4.8×, which turns a 150 × 40 px word box into roughly 30 × 8 px (hypothesis H1). We also check that every page has a box mask and a diff mask with the same base name, because the data adapter in P3 pairs files by name.

⏱ < 10 s. ✅ Pages about 2480 × 3508 px RGB; masks the same size, mode `L` or `1`; shrink factor ≈ 4.8; "missing" counts all 0.

In [ ]:
# PIL opens an image and reads its size without loading all the pixels
from PIL import Image

# Function: the file name without its extension (page_0001.png -> page_0001)
def base(name):
    return os.path.splitext(name)[0]

# Base names in each folder, as sets so we can compare them
names = {sub: {base(n) for n in os.listdir(os.path.join(SRC_RUN, sub))}
         for sub in ["tampered", "tampered_jpg", "originals", "masks/box", "masks/diff"]}
# How many tampered pages have no matching file in each other folder
for sub in ["tampered_jpg", "originals", "masks/box", "masks/diff"]:
    print(f"tampered pages with no match in {sub:13s}: {len(names['tampered'] - names[sub])}")

# Take the first 3 tampered pages
sample = sorted(os.listdir(os.path.join(SRC_RUN, "tampered")))[:3]
# One row per page: sizes of the page, its box mask and its original
rows = []
for n in sample:
    # Open the tampered page (only the header is read)
    page = Image.open(os.path.join(SRC_RUN, "tampered", n))
    # Find its box mask by base name (any extension)
    box_file = glob.glob(os.path.join(SRC_RUN, "masks/box", base(n) + ".*"))[0]
    box = Image.open(box_file)
    # How much ADCD-Net will shrink this page (longest side -> 512)
    shrink = max(page.size) / CONFIG["val_max_size"]
    # Save the facts for the table
    rows.append({"page": n, "page size (w x h)": page.size, "mode": page.mode,
                 "box mask size": box.size, "box mode": box.mode, "shrink factor": round(shrink, 2)})
# Show the table
print(pd.DataFrame(rows).to_string(index=False))

# Size of one JPEG copy and one original, to check they match the PNG
jpg_one = sorted(os.listdir(os.path.join(SRC_RUN, "tampered_jpg")))[0]
orig_one = sorted(os.listdir(os.path.join(SRC_RUN, "originals")))[0]
print("tampered_jpg example:", jpg_one, Image.open(os.path.join(SRC_RUN, "tampered_jpg", jpg_one)).size)
print("originals example   :", orig_one, Image.open(os.path.join(SRC_RUN, "originals", orig_one)).size)

**✅ P1 done when:** sections 1–5 ran without errors. Note anything marked ⚠️ and tell Claude Code (it will update CLAUDE.md facts and `notes/decisions.md`).

## 6. Build a separate Python 3.10 environment (P2)

**WHY:** ADCD-Net was written for Python 3.10 + PyTorch 2.3.0. Kaggle's own Python is newer, and mixing versions is the most common reason old research code breaks. So we build a separate, clean Python 3.10 "box" (a *virtual environment*, or venv) with `uv`, a fast installer. Think of it like running an old spreadsheet in the exact Excel version it was built in. The notebook itself keeps using Kaggle's Python; only the ADCD-Net scripts run inside the venv.

The helper `run()` defined here runs a shell command, prints only the last few lines, and on failure prints the **last 30 lines** (copy those to Claude Code) and stops. If the venv already exists (cell re-run), it is skipped.

⏱ 1–2 min. ✅ `OK (...)` lines, ending with `Python 3.10.x`.

In [ ]:
# Function: run one shell command, show a short log, stop with the last 30 lines if it fails
def run(cmd, show_tail=3):
    # Start a stopwatch
    t0 = time.time()
    # Run the command in a shell and capture everything it prints
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    # Join normal output and error output, split into lines
    lines = (res.stdout + res.stderr).strip().splitlines()
    # Seconds the command took
    secs = time.time() - t0
    # If it failed: print the last 30 lines and stop the cell
    if res.returncode != 0:
        print("\n".join(lines[-30:]))
        raise RuntimeError(f"FAILED after {secs:.0f}s: {cmd}")
    # If it worked: print the last few lines and the time
    if show_tail:
        print("\n".join(lines[-show_tail:]))
    print(f"OK ({secs:.0f}s): {cmd[:100]}")

# Keep uv's download cache in /tmp (not saved as notebook output)
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"
# Copy files instead of linking them (avoids a warning between different disks)
os.environ["UV_LINK_MODE"] = "copy"

# Install uv into Kaggle's Python
run("pip install -q uv", show_tail=0)
# Build the venv only if it is not there yet (resumable)
if not os.path.exists(VENV_PY):
    # uv downloads its own Python 3.10 and adds pip/setuptools/wheel (--seed)
    run(f"uv venv {VENV_DIR} --python {CONFIG['python_version']} --seed")
# Print the venv's Python version
run(f"{VENV_PY} --version", show_tail=1)

## 7. Install PyTorch 2.3.0 for CUDA 12.1 (P2)

**WHY:** PyTorch is the library ADCD-Net is built on. We install the exact version from the README (2.3.0, built for CUDA 12.1) from PyTorch's own download site. We also write a *constraints file* that pins `torch` and `numpy<2`, so later installs cannot quietly swap them for other versions (old compiled code like `jpeg2dct` breaks with numpy 2).

⏱ 3–6 min (about 2.5 GB download) — **good moment to start your lit review**. ✅ Last line like `torch 2.3.0+cu121 | CUDA 12.1 | GPU available: True`.

In [ ]:
# The constraints file: versions every later install must respect
CONSTRAINTS = "/tmp/constraints.txt"
with open(CONSTRAINTS, "w") as f:
    f.write("torch==2.3.0+cu121\ntorchvision==0.18.0+cu121\nnumpy<2\n")

# Install torch + torchvision (ds.py imports torchvision) from PyTorch's CUDA 12.1 index
run(f"uv pip install --python {VENV_PY} torch==2.3.0+cu121 torchvision==0.18.0+cu121 "
    f"--index-url https://download.pytorch.org/whl/cu121")
# Ask the venv's Python: torch version, CUDA version, can it see the GPU?
run(f"{VENV_PY} -c \"import torch; print('torch', torch.__version__, '| CUDA', torch.version.cuda, "
    f"'| GPU available:', torch.cuda.is_available())\"", show_tail=1)

## 8. Install the other packages, one at a time (P2)

**WHY:** the repo also imports `numpy`, `cv2` (OpenCV), `albumentations` (image resizing/augmentation, version 2.0.8 from the README), `lmdb`, `six`, `PIL`, `tqdm`, `einops`, `efficientnet_pytorch` and `tensorboard` (found with `grep` on the repo's imports). Installing **one at a time** means that if something fails, we know exactly which package caused it. Every install respects the constraints file from section 7. PaddleOCR is *not* installed here: the OCR masks come in P3 and will run in Kaggle's own Python.

⏱ 1–3 min. ✅ A table with `OK` for all 10 packages, then `numpy 1.26.x`.

In [ ]:
# Packages the ADCD-Net repo imports (numpy first, so nothing pulls in numpy 2)
PACKAGES = ["numpy<2", "opencv-python-headless", "albumentations==2.0.8", "lmdb", "six",
            "pillow", "tqdm", "einops", "efficientnet_pytorch", "tensorboard"]

# One row per package: name, status, seconds
install_log = []
for pkg in PACKAGES:
    # Stopwatch for this package
    t0 = time.time()
    # Install it into the venv, respecting the constraints file (stops the cell on failure)
    run(f"uv pip install --python {VENV_PY} -c {CONSTRAINTS} \"{pkg}\"", show_tail=0)
    # Record that it worked and how long it took
    install_log.append({"package": pkg, "status": "OK", "seconds": round(time.time() - t0, 1)})

# Show the install table
print(pd.DataFrame(install_log).to_string(index=False))
# Confirm numpy stayed below version 2
run(f"{VENV_PY} -c \"import numpy; print('numpy', numpy.__version__)\"", show_tail=1)

## 9. Build jpeg2dct (P2 — the risky one)

**WHY:** a JPEG file does not store pixels directly; it stores, for every 8×8 block, 64 "frequency" numbers (the DCT coefficients) — like describing a price series by its Fourier components instead of the daily prices. ADCD-Net reads these numbers straight from the file with `jpeg2dct` (made by Uber). It is written in C++, so it must be *compiled* on Kaggle, which needs the system JPEG library (`libjpeg-dev`) and a compiler. We checked its `setup.py`: the TensorFlow part is optional and skipped if TensorFlow is missing, so only the numpy part is built. `--no-build-isolation` makes it compile against the venv's numpy 1.26.

⏱ 1–3 min. ✅ `OK` lines and then `jpeg2dct OK: DCT shape (8, 8, 64)`.

🛑 **If this cell fails: STOP and paste the last 30 lines to Claude Code. Do not write a replacement** (CLAUDE.md rule: ask first).

In [ ]:
# Install the system JPEG library headers and a C/C++ compiler (Kaggle runs as root, so no sudo)
run("apt-get update -qq && apt-get install -y -qq libjpeg-dev build-essential", show_tail=0)
# Build tools the old setup.py needs inside the venv
run(f"uv pip install --python {VENV_PY} -c {CONSTRAINTS} \"setuptools<70\" wheel", show_tail=0)
# Compile and install jpeg2dct against the venv's numpy
run(f"uv pip install --python {VENV_PY} -c {CONSTRAINTS} --no-build-isolation jpeg2dct")

# Tiny test: save a random 64x64 grey JPEG and read its DCT numbers back
test_code = (
    "import numpy as np, tempfile; from PIL import Image; from jpeg2dct.numpy import load; "
    "p = tempfile.mktemp(suffix='.jpg'); "
    "Image.fromarray((np.random.rand(64, 64) * 255).astype('uint8')).save(p, quality=100); "
    "y, _, _ = load(p, normalized=False); print('jpeg2dct OK: DCT shape', y.shape)"
)
# Run the test with the venv's Python (64 px / 8 = 8 blocks per side, 64 numbers per block)
run(f"{VENV_PY} -c \"{test_code}\"", show_tail=1)

## 10. Clone the ADCD-Net code (P2)

**WHY:** the model definition (`model/`), the data loading (`ds.py`) and the settings (`cfg.py`) live in the authors' GitHub repo. We download it once into `/kaggle/working/ADCD-Net-code` and record the exact commit, so the run can be repeated with the same code. We never edit these files; we change settings from our own scripts instead.

⏱ < 15 s. ✅ A commit hash and the root files: `build_path_pkl.py, cfg.py, ds.py, fig, get_qt.py, loss, main.py, model, seg_char.py, utils.py` (plus README/LICENSE).

In [ ]:
# Clone only the latest version (depth 1), and only if not cloned already (resumable)
if not os.path.isdir(REPO_DIR):
    run(f"git clone -q --depth 1 {CONFIG['repo_url']} {REPO_DIR}", show_tail=0)
# Read the exact commit we got
commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Repo commit:", commit)
print("Root files :", sorted(os.listdir(REPO_DIR)))

# Save the commit in CONFIG and re-save config.json
CONFIG["repo_commit"] = commit
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

## 11. Final check: imports, versions and weights (P2)

**WHY:** this proves the whole environment works *before* we spend time on data. A small script (written by the `%%writefile` cell, run with the venv's Python from inside the repo folder so `import cfg` works) does three things:
1. prints the versions of Python, torch, numpy, OpenCV and albumentations, and whether CUDA works;
2. imports the repo's `ds` module and the `ADCDNet` class (if this works, every package the repo needs is installed);
3. opens both weight files and prints only their **key names and tensor counts** (never the weights themselves). `ADCDNet.pth` should contain a key `model`; `docres.pkl` a key `model_state`. We do not build the model yet — that is P4.

The output is also saved as `env_versions.txt` in the run folder.

⏱ 20–60 s (loading the weights takes most of it). ✅ Something like:
```
python 3.10.x | torch 2.3.0+cu121 | CUDA available True
numpy 1.26.x | cv2 4.x | albumentations 2.0.8 | jpeg2dct ok
import ds OK | import ADCDNet OK
ADCDNet.pth top-level keys: ['model', ...] | tensors in 'model': <a few hundred+>
docres.pkl  top-level keys: ['model_state', ...] | tensors in 'model_state': <a few hundred>
ALL CHECKS PASSED
```

In [ ]:
%%writefile /kaggle/working/adcd_scripts/check_env.py
# check_env.py: run with the venv Python from inside the repo folder.
# Usage: python check_env.py <path to ADCDNet.pth> <path to docres.pkl>

# Standard tools: command-line arguments and Python version
import sys, platform
# Read the two weight-file paths from the command line
ckpt_path, docres_path = sys.argv[1], sys.argv[2]

# Import the main libraries and print their versions
import torch, numpy, cv2, albumentations
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA available", torch.cuda.is_available())
# jpeg2dct has no version number to print, so just show it imports
from jpeg2dct.numpy import load
print("numpy", numpy.__version__, "| cv2", cv2.__version__, "| albumentations", albumentations.__version__, "| jpeg2dct ok")

# Make the repo folder importable (we are running from inside it)
sys.path.insert(0, ".")
# Import the repo's settings and point them at our weight files (in memory only, cfg.py is not edited)
import cfg
cfg.ckpt = ckpt_path
cfg.docres_ckpt_path = docres_path
# Import the data module and the model class
import ds
print("import ds OK", end=" | ")
from model.model import ADCDNet
print("import ADCDNet OK")

# Open the ADCD-Net checkpoint the same way main.py does (weights_only=True = safe loading)
ck = torch.load(ckpt_path, map_location="cpu", weights_only=True)
# Print only the key names and how many tensors are stored under 'model'
print("ADCDNet.pth top-level keys:", list(ck.keys())[:10], "| tensors in 'model':", len(ck["model"]))
# Same for the DocRes backbone, the way model.py loads it
dr = torch.load(docres_path, map_location="cpu", weights_only=True)
print("docres.pkl  top-level keys:", list(dr.keys())[:10], "| tensors in 'model_state':", len(dr["model_state"]))
print("ALL CHECKS PASSED")

In [ ]:
# Run the check script with the venv Python, from inside the repo folder, and capture its output
res = subprocess.run([VENV_PY, f"{SCRIPTS_DIR}/check_env.py", CKPT_PATH, DOCRES_PATH],
                     cwd=REPO_DIR, capture_output=True, text=True)
# Keep normal output plus only the last 30 lines of any errors/warnings
report = res.stdout + "\n".join(res.stderr.strip().splitlines()[-30:])
# Show it
print(report)
# Save it next to config.json so the versions are on record
with open(f"{RUN_DIR}/env_versions.txt", "w") as f:
    f.write(report)
print("Saved:", f"{RUN_DIR}/env_versions.txt", "| exit code:", res.returncode, "(0 = success)")

**✅ P2 done when** section 11 ends with `ALL CHECKS PASSED` and exit code 0.

**Then tell Claude Code:** "P1–P2 done" plus any ⚠️ lines from section 3, the table from section 5, and the section-11 output (it is short). It will log them in `notes/decisions.md` and add the P3 cells (data adapter + OCR masks) below this one.

**Note:** the venv lives in `/tmp`, so it disappears when the Kaggle session ends. That is fine: re-running sections 6–9 rebuilds it in ~10 min (and Save & Run All in P5 does this automatically).

# P3 — Data adapter and OCR masks

**WHY (big picture):** ADCD-Net's data loader (`GeneralValDs`) expects, for every image, a triple of files: *(image, answer-key mask, OCR text mask)*. The repo's own scripts hard-code someone else's folders, so we build the same folder layout ourselves for our 3 input variants:

| variant | images come from | answer-key mask |
|---|---|---|
| `png` | `tampered/*.png` | `masks/box/*.png` |
| `jpg90` | `tampered_jpg/*.jpg` | `masks/box/*.png` |
| `orig` | `originals/*.png` | all zeros (nothing was edited) |

| Section | Step | ⏱ Time | Where |
|---|---|---|---|
| 12. OCR environment (PaddleOCR) | P3 | 3–6 min | CPU |
| 13. Adapter folders: images + masks | P3 | ~30 s | CPU |
| 14. OCR script + **timing test on 3 pages** | P3 | 2–4 min (first run downloads the OCR model) | CPU |
| 15. OCR masks for all 300 images | P3 | **unknown until section 14 tells us** | CPU |
| 16. path_pkl files + qt_table.pk | P3 | < 15 s | CPU |
| 17. Figure + text-mask coverage check (H4) | P3 | ~20 s | CPU |

Needs the variables from sections 1–11 (`RUN_DIR`, `SRC_RUN`, `VENV_PY`, `run()`, ...). If you restarted the kernel, re-run sections 1, 3 and the `run()` definition cell in section 6 first (not the installs).

## 12. A second environment for PaddleOCR (P3)

**WHY:** the OCR text mask tells ADCD-Net *where the text is*, so it can compare text pixels with clean background pixels. The authors make it with PaddleOCR (PP-OCRv5 text *detector*: it draws a box around every text line/word, it does not read the letters). PaddleOCR and PyTorch can fight over the GPU, so we give PaddleOCR its **own** small Python 3.10 environment and run it on the **CPU**, as its own process. Packages are installed one at a time, and `numpy<2` is pinned, as before.

⏱ 3–6 min. ✅ `OK` lines, then a line like `paddle 3.x.x | paddleocr 3.x.x | cv2 4.x`. If a version differs, that's fine; it is logged. If this fails, paste the last 30 lines.

In [ ]:
# Where the OCR environment lives (in /tmp, so it is not saved as notebook output)
OCR_VENV = "/tmp/venv_ocr"
# Its Python interpreter
OCR_PY = f"{OCR_VENV}/bin/python"

# Create the venv only if it does not exist yet (resumable)
if not os.path.exists(OCR_PY):
    run(f"uv venv {OCR_VENV} --python {CONFIG['python_version']} --seed")
# Install the packages one at a time, numpy first so nothing pulls numpy 2
for pkg in ["numpy<2", "paddlepaddle", "paddleocr"]:
    run(f"uv pip install --python {OCR_PY} \"{pkg}\"", show_tail=0)

# Ask the OCR venv for the versions it ended up with
check = ("import paddle, paddleocr, cv2; "
         "print('paddle', paddle.__version__, '| paddleocr', paddleocr.__version__, '| cv2', cv2.__version__)")
run(f"{OCR_PY} -c \"{check}\"", show_tail=1)

## 13. Adapter folders: images and answer-key masks (P3)

**WHY:** for each variant we make `inputs/<variant>/images/`, `masks/` and `ocr/` (the OCR masks arrive in sections 14–15). Rules from the repo: the mask name is the image name with `.jpg` replaced by `.png`; the OCR mask has the *same* file name as the image. Pages are huge, so `images/` holds **links** to the read-only Kaggle dataset instead of copies (no extra disk use). Box masks are small, so they are copied. For originals we write all-zero masks of the right size.

The cell also checks that every mask has exactly the same width and height as its page. If they differ, the loader would crash or mis-align the answer key.

⏱ ~30 s. ✅ `png: 100 images, 100 masks` for all three variants, `size mismatches: 0`, and a note on disk used.

In [ ]:
# numpy for the all-zero masks, PIL to read image sizes and write PNGs
from PIL import Image

# Folder that will hold the three adapter folders
INPUTS_DIR = f"{RUN_DIR}/inputs"

# Recipe for each variant: (folder with the images, image extension, where the mask comes from)
VARIANTS = {
    "png":   {"img_dir": f"{SRC_RUN}/tampered",     "ext": ".png", "mask": "box"},
    "jpg90": {"img_dir": f"{SRC_RUN}/tampered_jpg", "ext": ".jpg", "mask": "box"},
    "orig":  {"img_dir": f"{SRC_RUN}/originals",    "ext": ".png", "mask": "zeros"},
}

# Count size mismatches between a page and its mask
n_mismatch = 0
# One variant at a time
for variant, recipe in VARIANTS.items():
    # Make the three sub-folders for this variant (no error if they already exist)
    for sub in ["images", "masks", "ocr"]:
        os.makedirs(f"{INPUTS_DIR}/{variant}/{sub}", exist_ok=True)
    # All image files of this variant, sorted
    files = sorted(os.listdir(recipe["img_dir"]))
    for fname in files:
        # Path of the real image in the Kaggle dataset
        src_img = f"{recipe['img_dir']}/{fname}"
        # Path of its link inside our adapter folder
        dst_img = f"{INPUTS_DIR}/{variant}/images/{fname}"
        # Make the link only if it is not there yet (resumable)
        if not os.path.lexists(dst_img):
            os.symlink(src_img, dst_img)
        # Mask file name = image name with .jpg replaced by .png (repo rule)
        mask_name = fname.replace(".jpg", ".png")
        # Where our mask goes
        dst_mask = f"{INPUTS_DIR}/{variant}/masks/{mask_name}"
        # Page size (width, height), read from the file header only
        page_size = Image.open(src_img).size
        if recipe["mask"] == "box":
            # Real box mask: the file has the same base name as the page
            src_mask = f"{SRC_RUN}/masks/box/{os.path.splitext(fname)[0]}.png"
            # Copy it once
            if not os.path.exists(dst_mask):
                Image.open(src_mask).convert("L").save(dst_mask)
            # Check that the mask and the page have the same size
            if Image.open(dst_mask).size != page_size:
                n_mismatch += 1
        else:
            # Originals: nothing was edited, so the mask is all zeros (height x width)
            if not os.path.exists(dst_mask):
                zeros = np.zeros((page_size[1], page_size[0]), dtype=np.uint8)
                Image.fromarray(zeros).save(dst_mask)
    # Report the counts for this variant
    print(f"{variant:6s}: {len(os.listdir(f'{INPUTS_DIR}/{variant}/images'))} images, "
          f"{len(os.listdir(f'{INPUTS_DIR}/{variant}/masks'))} masks")

print("size mismatches (page vs mask):", n_mismatch)
# Disk used by the masks (images are links, so they cost nothing)
print("Disk used by inputs/ (masks only):", subprocess.run(["du", "-sh", INPUTS_DIR], capture_output=True, text=True).stdout.split()[0])

## 14. OCR script and a timing test on 3 pages (P3)

**WHY:** this is our own copy of the repo's `seg_char.py` logic (we do not edit the repo): run the PP-OCRv5 *server* text detector on a page, keep every detected polygon with score **> 0.5**, fill them in white (255) on a black page-sized image, save as PNG. The script:
- processes a list of images and **skips any mask that already exists**, so it can be stopped and restarted;
- writes each file to a temporary name first, then renames it, so a crash never leaves a half-written mask;
- writes **PNG data even into `.jpg`-named files** (the loader reads by content, and this avoids JPEG noise creating fake text pixels);
- records seconds per page in `ocr_timing.csv`.

We first run only **3 pages** to measure the speed and make the model download happen once. The cell then estimates the time for all 300 images.

⏱ 2–4 min (the model downloads the first time; the 3 pages themselves should take a few seconds each, we'll see how many). ✅ `model loaded in ...s`, three timing lines, and an estimate for the full run in minutes. **Write the estimate down.**

In [ ]:
%%writefile /kaggle/working/adcd_scripts/ocr_masks.py
# ocr_masks.py: make PaddleOCR text masks, one PNG per image. Run with the OCR venv's Python.
# Usage: python ocr_masks.py <manifest.json> <timing.csv> <limit (0 = all)>

# Standard tools
import sys, os, json, time, csv
# Stop PaddleOCR from checking model-hosting sites at start-up (saves seconds)
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")
# Image tools and the PaddleOCR text detector
import cv2
import numpy as np
from paddleocr import TextDetection

# Read the command-line arguments
manifest_path, timing_csv, limit = sys.argv[1], sys.argv[2], int(sys.argv[3])
# The manifest is a list of [variant, image_path, mask_output_path]
items = json.load(open(manifest_path))
# Keep only the masks that do not exist yet (this makes the script resumable)
todo = [it for it in items if not os.path.exists(it[2])]
print(f"{len(items)} images in manifest, {len(todo)} masks still to make", flush=True)
# If a limit was given, only do that many (used for the timing test)
if limit > 0:
    todo = todo[:limit]

# Load the text detector (same model name as the repo's seg_char.py).
# enable_mkldnn=False switches off Intel's oneDNN speed-up: in this Paddle version it crashes
# on CPU (ConvertPirAttribute2RuntimeAttribute error). Results are the same, only slower.
t0 = time.time()
model = TextDetection(model_name="PP-OCRv5_server_det", enable_mkldnn=False)
print(f"model loaded in {time.time() - t0:.1f}s", flush=True)

# Write the timing-file header the first time
if not os.path.exists(timing_csv):
    with open(timing_csv, "w", newline="") as f:
        csv.writer(f).writerow(["variant", "image", "seconds", "n_text_regions"])

# Go through the images one by one
for i, (variant, img_path, out_path) in enumerate(todo, 1):
    # Stopwatch for this page
    t = time.time()
    # Detect text regions (batch of 1, like the repo)
    res = model.predict(input=img_path, batch_size=1)[0]
    # Polygons around the text and the confidence of each polygon
    polys, scores = res["dt_polys"], res["dt_scores"]
    # Read the page only to get its height and width
    h, w = cv2.imread(img_path).shape[:2]
    # Start with an all-black mask
    mask = np.zeros((h, w), dtype=np.uint8)
    # Number of polygons kept
    kept = 0
    # Fill every polygon whose score is above 0.5 (repo rule)
    for poly, score in zip(polys, scores):
        if score > 0.5:
            cv2.fillPoly(mask, [poly.astype(np.int32)], 1)
            kept += 1
    # Encode as PNG (0/255), even if the file name ends in .jpg
    ok, buf = cv2.imencode(".png", mask * 255)
    # Write to a temporary name, then rename: no half-written files after a crash
    with open(out_path + ".tmp", "wb") as f:
        f.write(buf.tobytes())
    os.replace(out_path + ".tmp", out_path)
    # Seconds this page took
    secs = time.time() - t
    # Log it
    with open(timing_csv, "a", newline="") as f:
        csv.writer(f).writerow([variant, os.path.basename(img_path), round(secs, 2), kept])
    # Print progress for the first 3 pages, then every 10th
    if i <= 3 or i % 10 == 0 or i == len(todo):
        print(f"[{i}/{len(todo)}] {variant}/{os.path.basename(img_path)}  {secs:.1f}s  regions kept: {kept}", flush=True)
print("done", flush=True)

In [ ]:
# Function: run a command and print its output as it arrives (for long jobs)
def run_stream(cmd_list, env=None):
    # Start the process, merging error text into normal text
    proc = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    # Keep every line so the last 30 can be shown on failure
    lines = []
    # Print each line the moment it arrives
    for line in proc.stdout:
        print(line, end="")
        lines.append(line)
    # Wait for the process to end
    proc.wait()
    # If it failed, show the last 30 lines and stop
    if proc.returncode != 0:
        print("".join(lines[-30:]))
        raise RuntimeError("OCR script failed (see the last 30 lines above)")

# Manifest = every image of every variant with the path where its OCR mask must go
manifest = []
for variant in VARIANTS:
    for fname in sorted(os.listdir(f"{INPUTS_DIR}/{variant}/images")):
        manifest.append([variant, f"{INPUTS_DIR}/{variant}/images/{fname}", f"{INPUTS_DIR}/{variant}/ocr/{fname}"])
# Save the manifest so the OCR process can read it
MANIFEST_PATH = f"{RUN_DIR}/ocr_manifest.json"
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f)
# Where the per-page seconds are logged
OCR_TIMING = f"{RUN_DIR}/ocr_timing.csv"

# Run the OCR script on just 3 pages (limit = 3)
run_stream([OCR_PY, f"{SCRIPTS_DIR}/ocr_masks.py", MANIFEST_PATH, OCR_TIMING, "3"])

# Read the timing log and estimate the full run
timing = pd.read_csv(OCR_TIMING)
# Median is robust: the first page is slower because of warm-up
sec_per_page = timing["seconds"].median()
# How many masks are still missing
remaining = sum(1 for _, _, out in manifest if not os.path.exists(out))
print(f"\nMedian {sec_per_page:.1f} s/page | {remaining} masks left | estimate for the rest: {sec_per_page * remaining / 60:.0f} min")

## 15. OCR masks for all 300 images (P3)

**WHY:** every image needs its own text mask (tampered pages have a repainted word, so their text boxes can differ slightly from the original's). This is the only slow part of P3.

**Safety rule from CLAUDE.md:** anything over 30 minutes needs your OK. The cell below **stops by itself** if the estimate from section 14 is above 30 minutes. If it stops, **tell Claude Code the estimate** (don't just flip the switch): we'll decide whether to wait, run it in the background while you do your lit review, or speed it up. If the estimate is under 30 minutes it just runs.

It is **resumable**: if the session stops, re-run this cell and it continues where it left off.

⏱ **= the estimate printed in section 14** (it prints progress every 10 pages). ✅ ends with `done` and `300 / 300 OCR masks exist`.

In [ ]:
# Set this to True ONLY after talking to Claude Code about a run longer than 30 minutes
CONFIRM_LONG_OCR = True

# Re-count the masks that are still missing
remaining = sum(1 for _, _, out in manifest if not os.path.exists(out))
# Re-read the timing log (median seconds per page)
sec_per_page = pd.read_csv(OCR_TIMING)["seconds"].median()
# Estimated minutes for what is left
est_min = sec_per_page * remaining / 60
print(f"{remaining} masks left, about {est_min:.0f} min at {sec_per_page:.1f} s/page")

# Stop if it is too long and I have not agreed yet
if est_min > 30 and not CONFIRM_LONG_OCR:
    raise SystemExit("STOP: estimate is over 30 min. Tell Claude Code the estimate before continuing.")

# Run the OCR script on everything that is missing (limit 0 = no limit)
run_stream([OCR_PY, f"{SCRIPTS_DIR}/ocr_masks.py", MANIFEST_PATH, OCR_TIMING, "0"])

# Final count
done = sum(1 for _, _, out in manifest if os.path.exists(out))
print(f"\n{done} / {len(manifest)} OCR masks exist")
# Timing summary per variant
print(pd.read_csv(OCR_TIMING).groupby("variant")["seconds"].agg(["count", "mean", "max"]).round(1))

## 16. Path lists (.pkl) and the quantization table (P3)

**WHY (path lists):** `GeneralValDs` does not scan folders; it loads a pickle file with a list of `(image_path, mask_path, ocr_path)` tuples. This is what `build_path_pkl.py` does; we do the same for each variant, and check that every file exists.

**WHY (qt table):** a JPEG stores each 8×8 block of frequencies divided by a *quantization table* (a table of 64 "rounding step sizes"; bigger steps = more detail lost). ADCD-Net also receives this table as input. The authors keep a file `qt_table.pk` (QF → 8×8 table) inside the DocTamper data; **we did not upload it**, so we build the same kind of dictionary with Pillow: save a blank image at each quality factor (QF) 1–100 and read the table back with the repo's own `get_luma_qt_8x8`. For our run only QF 100 is used (the loader re-saves every page at quality 100). At QF 100 the steps are all 1, meaning almost nothing is rounded away.

⏱ < 15 s. ✅ Three `.pkl` files with 100 tuples each, `0 missing files`, and `qt_table.pk: built with Pillow (DocTamper file not used)` with QF 100 showing a table of all 1s.

In [ ]:
# pickle writes Python objects to a file; io gives an in-memory file; sys lets us import the repo's get_qt.py
import pickle, io, sys

# Folder for the three path lists
PKL_DIR = f"{RUN_DIR}/path_pkl"
os.makedirs(PKL_DIR, exist_ok=True)

# One path list per variant
for variant in VARIANTS:
    # Tuples of (image path, mask path, ocr path)
    path_list = []
    # Count files that do not exist
    n_missing = 0
    for fname in sorted(os.listdir(f"{INPUTS_DIR}/{variant}/images")):
        # Image, mask (.jpg replaced by .png) and OCR mask (same name as the image)
        img_p = f"{INPUTS_DIR}/{variant}/images/{fname}"
        mask_p = f"{INPUTS_DIR}/{variant}/masks/{fname.replace('.jpg', '.png')}"
        ocr_p = f"{INPUTS_DIR}/{variant}/ocr/{fname}"
        # Check that all three files exist
        if not (os.path.exists(img_p) and os.path.exists(mask_p) and os.path.exists(ocr_p)):
            n_missing += 1
            continue
        # Keep the triple
        path_list.append((img_p, mask_p, ocr_p))
    # Save the list
    with open(f"{PKL_DIR}/{variant}.pkl", "wb") as f:
        pickle.dump(path_list, f)
    print(f"{variant:6s}.pkl: {len(path_list)} tuples, {n_missing} missing files")

# Make the repo's get_qt.py importable (it only needs PIL and numpy)
sys.path.insert(0, REPO_DIR)
from get_qt import get_luma_qt_8x8

# Dictionary: quality factor -> 8x8 table
qt_dict = {}
for qf in range(1, 101):
    # Save a small blank grey image as JPEG at this quality, in memory
    buf = io.BytesIO()
    Image.new("L", (64, 64), 128).save(buf, format="JPEG", quality=qf)
    buf.seek(0)
    # Read the luma quantization table back as an 8x8 array
    qt_dict[qf] = get_luma_qt_8x8(Image.open(buf))

# Save it where our own scripts will point cfg.qt_path
QT_PATH = f"{RUN_DIR}/qt_table.pk"
with open(QT_PATH, "wb") as f:
    pickle.dump(qt_dict, f)
print("\nqt_table.pk: built with Pillow (DocTamper file not used) ->", QT_PATH)
print("QF 100 table unique values:", np.unique(qt_dict[100]).tolist(), "| QF 75 top-left row:", qt_dict[75][0].tolist())

## 17. Figure: page, OCR mask, box mask + text-mask coverage (P3)

**WHY:** a quick look to confirm the adapter is right (masks line up with the page) and a first test of hypothesis **H4** (*the OCR mask may miss small or digit-only words*).

**Coverage** = the share of the box pixels that are also white in the OCR mask:
- WHAT: how much of the edited word the text mask "knows about".
- WHY: ADCD-Net is told where text is only through this mask; if the box is outside it, the model gets no text hint exactly where the edit is.
- HOW/formula: `coverage = (box pixels that are also OCR pixels) / (box pixels)`.
- Tiny example: the box has 100 pixels and 60 of them are inside the OCR mask → coverage = 0.60.

The figure shows pages 1–3 of the `png` variant shrunk for display, with the box outlined in red on the OCR mask. The printed summary covers all 100 forgeries. Nothing here is the model yet.

⏱ ~20 s. ✅ A 3×3 figure saved as `figures/adapter_examples.png`, and a summary of coverage (min / median / max) and the number of blank OCR masks. **If an OCR mask is blank or the box lies outside it, write that down: it is evidence for H4.**

In [ ]:
# matplotlib draws the figure
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Function: load a picture as a grey numpy array
def load_grey(path):
    return np.array(Image.open(path).convert("L"))

# Coverage of the box by the OCR mask, for all 100 forgeries (png variant)
cover_rows = []
for fname in sorted(os.listdir(f"{INPUTS_DIR}/png/images")):
    # Box mask and OCR mask as True/False arrays
    box = load_grey(f"{INPUTS_DIR}/png/masks/{fname}") > 0
    ocr = load_grey(f"{INPUTS_DIR}/png/ocr/{fname}") > 0
    # Share of box pixels that are also OCR pixels
    coverage = (box & ocr).sum() / max(box.sum(), 1)
    # Share of the whole page the OCR mask covers (0 = blank mask)
    ocr_share = ocr.mean()
    cover_rows.append({"image": fname, "box_pixels": int(box.sum()), "coverage": coverage, "ocr_page_share": ocr_share})
cover = pd.DataFrame(cover_rows)
# Save the table (used again in notebook 06 for the text-mask test)
cover.to_csv(f"{RUN_DIR}/ocr_box_coverage.csv", index=False)

# Summary numbers
print("Box coverage by OCR mask (100 forgeries):")
print("  min / median / max :", round(cover["coverage"].min(), 2), "/", round(cover["coverage"].median(), 2), "/", round(cover["coverage"].max(), 2))
print("  boxes with 0 coverage     :", int((cover["coverage"] == 0).sum()))
print("  boxes covered under 50%   :", int((cover["coverage"] < 0.5).sum()))
print("  blank OCR masks           :", int((cover["ocr_page_share"] == 0).sum()))
print("  OCR mask share of the page: median", round(cover["ocr_page_share"].median() * 100, 1), "%")

# Figure: 3 pages (rows) x 3 panels (columns)
fig, axes = plt.subplots(3, 3, figsize=(10, 14))
for row, fname in enumerate(sorted(os.listdir(f"{INPUTS_DIR}/png/images"))[:3]):
    # Load page, OCR mask and box mask
    page = Image.open(f"{INPUTS_DIR}/png/images/{fname}").convert("RGB")
    ocr = Image.open(f"{INPUTS_DIR}/png/ocr/{fname}").convert("L")
    box = Image.open(f"{INPUTS_DIR}/png/masks/{fname}").convert("L")
    # Shrink all three to the same display size (longest side 600 px)
    scale = 600 / max(page.size)
    small = (int(page.size[0] * scale), int(page.size[1] * scale))
    page, ocr, box = page.resize(small), ocr.resize(small), box.resize(small)
    # Bounding rectangle of the box mask (for the red outline)
    ys, xs = np.nonzero(np.array(box))
    # Coverage value for this page
    cov = cover.loc[cover["image"] == fname, "coverage"].iloc[0]
    # Panel 1: the page
    axes[row, 0].imshow(page)
    axes[row, 0].set_title(f"{fname}: tampered page")
    # Panel 2: OCR mask with the box outlined in red
    axes[row, 1].imshow(ocr, cmap="gray")
    axes[row, 1].add_patch(plt.Rectangle((xs.min(), ys.min()), xs.max() - xs.min(), ys.max() - ys.min(),
                                         fill=False, edgecolor="red", linewidth=1.5))
    axes[row, 1].set_title(f"OCR text mask (box covered: {cov:.0%})")
    # Panel 3: the answer-key box mask
    axes[row, 2].imshow(box, cmap="gray")
    axes[row, 2].set_title("Box mask (answer key)")
    # Label the axes in display pixels
    for col in range(3):
        axes[row, col].set_xlabel("x (display pixels)")
        axes[row, col].set_ylabel("y (display pixels)")
fig.suptitle("Adapter check: page, OCR text mask, answer-key box", y=0.995)
plt.tight_layout()
# Save the figure as PNG
plt.savefig(f"{RUN_DIR}/figures/adapter_examples.png", dpi=120)
plt.show()

# P4 — Smoke test and sanity check

**WHY (big picture):** the repo has no "predict" script, only a multi-GPU evaluation inside `main.py`. So we write our own **single-GPU loop** that copies exactly what `main.py` does in evaluation: the same data loader (`GeneralValDs`), the same padding (`pad_collate`: pad to a square that is a multiple of 16), the same float16 mode (autocast), and the same crop back to the real image size. Then we test it twice:
1. **Smoke test** on 3 of our pages: does it run, how fast, and do the maps look sensible?
2. **Sanity check** on a *published* test set (T-SROIE) where the paper reports a score. If our F1 there is close to the paper's, the setup is right, and a low score on bank statements is a real finding, not a bug.

| Section | ⏱ Time | GPU? |
|---|---|---|
| 18. Backup of the P3 outputs (OCR masks took 42 min!) | < 30 s | no |
| 19. Write the inference script | < 1 s | no |
| 20. Smoke test on 3 pages + overlay figure | 1–2 min (model loading) | yes |
| 21. Sanity check: check size, download T-SROIE | 1–5 min (stops if > 2 GB) | no |
| 22. Sanity check: OCR masks + path list | 1–10 min (only if the set has no OCR masks; stops if > 30 min) | no |
| 23. Sanity check: run ADCD-Net, compare with the paper | 1–3 min | yes |

Needs the variables from sections 1–17 in this session.

## 18. Backup of the P3 outputs (P4)

**WHY:** the OCR masks took 42 minutes, and Kaggle deletes `/kaggle/working` when the session ends. This cell zips the small, hard-to-remake files (OCR masks, path lists, qt table, timing and coverage tables, figures, config) into one file. The page images are **not** included (they are links to your dataset).

**After it runs: download `/kaggle/working/adcd_p3_backup.zip`** (right panel → Output → `...` → Download). If the session dies, this zip can be uploaded as a Kaggle Dataset so we never redo the OCR.

⏱ < 30 s. ✅ `300 OCR masks + ... files zipped`, size a few MB.

In [ ]:
# zipfile writes .zip archives
import zipfile

# Where the backup goes (outside the run folder, easy to find)
BACKUP_ZIP = f"{WORK}/adcd_p3_backup.zip"
# Count what we put in
n_ocr, n_other = 0, 0
# Open a new zip file for writing (compressed)
with zipfile.ZipFile(BACKUP_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    # Every OCR mask of every variant, stored as <run name>/inputs/<variant>/ocr/<file>
    for variant in VARIANTS:
        for fname in sorted(os.listdir(f"{INPUTS_DIR}/{variant}/ocr")):
            zf.write(f"{INPUTS_DIR}/{variant}/ocr/{fname}", f"{RUN_NAME}/inputs/{variant}/ocr/{fname}")
            n_ocr += 1
    # The small files and folders at the top of the run folder
    for item in ["config.json", "env_versions.txt", "ocr_timing.csv", "ocr_box_coverage.csv", "qt_table.pk", "path_pkl", "figures"]:
        path = f"{RUN_DIR}/{item}"
        # A single file: add it
        if os.path.isfile(path):
            zf.write(path, f"{RUN_NAME}/{item}")
            n_other += 1
        # A folder: add every file inside it
        elif os.path.isdir(path):
            for fname in sorted(os.listdir(path)):
                zf.write(f"{path}/{fname}", f"{RUN_NAME}/{item}/{fname}")
                n_other += 1
# Report
print(f"{n_ocr} OCR masks + {n_other} other files zipped -> {BACKUP_ZIP} "
      f"({os.path.getsize(BACKUP_ZIP) / 1e6:.1f} MB). Download it now.")

In [ ]:
import os
from pathlib import Path
from IPython.display import FileLink, display

zip_path = Path("/kaggle/working/adcd_p3_backup.zip")
previous_directory = os.getcwd()

try:
    if zip_path.is_file():
        os.chdir(zip_path.parent)
        display(FileLink(
            zip_path.name,
            result_html_prefix="Download your backup ZIP: "
        ))
    else:
        print(f"File not found: {zip_path}")
finally:
    os.chdir(previous_directory)


## 19. The inference script (P4)

**WHY:** this script is the heart of the evaluation. It runs with the **venv's** Python (3.10 + torch 2.3) on **one GPU**. For each image in a path list it:
1. lets the repo's `GeneralValDs` do the official preprocessing (shrink to longest side 512 → greyscale → re-save as JPEG q100 → read DCT numbers → QT table for QF 100);
2. pads with the repo's `pad_collate`, runs the model in float16 like `main.py`, and crops back to the real size;
3. turns the 2 output scores per pixel into a probability with **softmax** (class 1 = "tampered") and saves it as a float16 `.npy` map (at the model's resolution, ≤ 512 px);
4. writes one row per image to a CSV: seconds, max probability, 99.9th percentile, mean of the top 0.1 % of pixels, and the **official-style precision / recall / F1** computed exactly like `main.py` (prediction = argmax, i.e. probability ≥ 0.5).

It skips images whose `.npy` already exists (resumable). It also prints how many checkpoint weights did **not** match the model ("missing" / "unexpected" keys). Both should be 0 or close to it; a large number would mean the weights did not load.

⏱ < 1 s (it only writes the file).

In [ ]:
%%writefile /kaggle/working/adcd_scripts/infer.py
# infer.py: run pretrained ADCD-Net on one path list and save tamper-probability maps.
# Run with the venv Python. One GPU, no DDP. Copies the evaluation steps of the repo's main.py.

# Standard tools
import argparse, os, sys, time, csv
import numpy as np
import torch

# Read the command-line options
ap = argparse.ArgumentParser()
# Folder of the ADCD-Net repo
ap.add_argument("--repo")
# Folder with the <ds_name>.pkl path lists, and which list to run
ap.add_argument("--pkl_dir")
ap.add_argument("--ds_name")
# Where to save the .npy maps and the per-image CSV
ap.add_argument("--out_dir")
ap.add_argument("--csv")
# Weight files and the quantization-table file
ap.add_argument("--ckpt")
ap.add_argument("--docres")
ap.add_argument("--qt")
# Longest side after shrinking (official: 512), max images to do (0 = all), seed
ap.add_argument("--max_size", type=int, default=512)
ap.add_argument("--limit", type=int, default=0)
ap.add_argument("--seed", type=int, default=42)
args = ap.parse_args()

# Work from inside the repo so its own imports (cfg, ds, model) work
os.chdir(args.repo)
sys.path.insert(0, args.repo)
# Point the repo's settings at our files (in memory only; cfg.py is not edited)
import cfg
cfg.ckpt = args.ckpt
cfg.docres_ckpt_path = args.docres
cfg.pkl_dir = args.pkl_dir
cfg.qt_path = args.qt
cfg.val_max_size = args.max_size
# The repo's data loader, its padding function, and the model class
from ds import GeneralValDs, pad_collate
from model.model import ADCDNet
# float16 mode, exactly as main.py uses in evaluation
from torch.amp import autocast
from torch.utils.data import DataLoader, Subset

# Seeds (the evaluation itself has no randomness, but this is our rule)
torch.manual_seed(args.seed)
np.random.seed(args.seed)
# Small number to avoid dividing by zero (same as main.py)
EPS = 1e-8

# Build the dataset from the path list
ds = GeneralValDs(args.ds_name)
# Image names exactly as the loader makes them (file name up to the first dot)
names = [os.path.basename(p[0]).split(".")[0] for p in ds.path_list]
# Make the output folder
os.makedirs(args.out_dir, exist_ok=True)
# Only the images without a saved map (resumable)
todo = [i for i, n in enumerate(names) if not os.path.exists(f"{args.out_dir}/{n}.npy")]
# Apply the limit if one was given
if args.limit > 0:
    todo = todo[:args.limit]
print(f"{len(names)} images in {args.ds_name}, {len(todo)} to do", flush=True)
# Nothing to do: stop here
if not todo:
    sys.exit(0)
# One image at a time, padded the official way
dl = DataLoader(Subset(ds, todo), batch_size=1, shuffle=False, num_workers=0, collate_fn=pad_collate)

# Build the model (this also loads docres.pkl) and move it to the GPU
t0 = time.time()
model = ADCDNet().cuda()
# Load the ADCD-Net weights the way main.py does (remove the 'module.' prefix left by multi-GPU training)
state = torch.load(args.ckpt, map_location="cpu", weights_only=True)["model"]
state = {k.replace("module.", ""): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
# Evaluation mode (no training behaviour)
model.eval()
print(f"model ready in {time.time() - t0:.1f}s | missing keys: {len(missing)} | unexpected keys: {len(unexpected)}", flush=True)
# Show a few key names if anything did not match
if missing or unexpected:
    print("  first missing:", missing[:5], "| first unexpected:", unexpected[:5], flush=True)

# Write the CSV header the first time
cols = ["image", "h", "w", "seconds", "max_prob", "p999", "top01_mean", "mean_prob",
        "precision", "recall", "f1", "gt_pixels", "pred_pixels"]
if not os.path.exists(args.csv):
    with open(args.csv, "w", newline="") as f:
        csv.writer(f).writerow(cols)

# No gradients needed for prediction
with torch.no_grad():
    # Stopwatch (includes loading + preprocessing of each image)
    t_last = time.time()
    for k, items in enumerate(dl, 1):
        # The 5 tensors the model needs, on the GPU (the true mask is passed but not used in eval mode)
        img, dct, qt, mask, ocr_mask = [x.cuda() for x in items[:5]]
        # Image name and its real size before padding
        name = items[5][0]
        h, w = items[6][0].tolist()
        # Run the model in float16 and keep the first output (the 2-class scores)
        with autocast(device_type="cuda", dtype=torch.float16):
            logits = model(img, dct, qt, mask, ocr_mask, is_train=False)[0]
        # Crop away the padding, back to float32
        logit = logits[0, :, :h, :w].float()
        # Probability of class 1 ("tampered") per pixel
        prob = torch.softmax(logit, dim=0)[1]
        # Official prediction = the class with the higher score (same as probability >= 0.5)
        pred = logit.argmax(0)
        # The answer key at the same size (1 = tampered)
        y = (mask[0, 0, :h, :w] == 1).long()
        # Official-style precision, recall and F1 for this image (main.py compute_f1)
        matched = (pred * y).sum().item()
        p = matched / (pred.sum().item() + EPS)
        r = matched / (y.sum().item() + EPS)
        f1 = 2 * p * r / (p + r + EPS)
        # Save the probability map as float16
        prob_np = prob.cpu().numpy()
        np.save(f"{args.out_dir}/{name}.npy", prob_np.astype(np.float16))
        # Image-level scores: max, 99.9th percentile, mean of the top 0.1 % of pixels, overall mean
        flat = np.sort(prob_np.ravel())
        n_top = max(1, int(np.ceil(0.001 * flat.size)))
        top01 = float(flat[-n_top:].mean())
        # Seconds for this image
        torch.cuda.synchronize()
        secs = time.time() - t_last
        t_last = time.time()
        # Append the row to the CSV
        row = [name, h, w, round(secs, 3), float(flat[-1]), float(np.percentile(flat, 99.9)), top01,
               float(flat.mean()), p, r, f1, int(y.sum().item()), int(pred.sum().item())]
        with open(args.csv, "a", newline="") as f:
            csv.writer(f).writerow(row)
        # Progress: first 3 images, then every 25th, and the last one
        if k <= 3 or k % 25 == 0 or k == len(todo):
            print(f"[{k}/{len(todo)}] {name}  {secs:.2f}s  max prob {flat[-1]:.3f}  F1 {f1:.3f}", flush=True)
print("done", flush=True)

## 20. Smoke test on 3 tampered pages + overlay figure (P4)

**WHY:** before running 300 images, check on 3 that (a) the script runs end to end, (b) the weights load (missing/unexpected keys ≈ 0), (c) the speed is OK, and (d) the maps are not all zero. The figure shows, for each page **as the model sees it** (shrunk to ≤ 512 px, grey): the box mask, the probability map (0 = surely clean, 1 = surely tampered) and the map after the 0.5 cut.

What to expect: our edits are small and different from the training data, so the maps may be faint or in the wrong place. **That is fine here**; this step only checks that the machinery works. Whether it "works" on documents is the job of the sanity check.

⏱ 1–2 min (most of it building the model). ✅ `missing keys: 0 | unexpected keys: 0` (or a small number), 3 progress lines, a table with `max_prob` not all 0, seconds per image (expect < 1 s after the first), an estimate for 300 images, and `figures/smoke_overlay.png`.

In [ ]:
# Folders for the maps and the per-image score tables
PROBS_DIR = f"{RUN_DIR}/probs"
SCORES_DIR = f"{RUN_DIR}/scores"
os.makedirs(SCORES_DIR, exist_ok=True)
# Use one GPU only (our loop is single-process)
GPU_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0")

# Function: build the command that runs infer.py on one path list
def infer_cmd(ds_name, out_name, limit=0):
    return [VENV_PY, f"{SCRIPTS_DIR}/infer.py",
            "--repo", REPO_DIR, "--pkl_dir", PKL_DIR, "--ds_name", ds_name,
            "--out_dir", f"{PROBS_DIR}/{out_name}", "--csv", f"{SCORES_DIR}/{out_name}.csv",
            "--ckpt", CKPT_PATH, "--docres", DOCRES_PATH, "--qt", QT_PATH,
            "--max_size", str(CONFIG["val_max_size"]), "--limit", str(limit), "--seed", str(CONFIG["seed"])]

# Run on the first 3 tampered PNG pages
run_stream(infer_cmd("png", "png", limit=3), env=GPU_ENV)

# Read the 3 result rows
smoke = pd.read_csv(f"{SCORES_DIR}/png.csv").head(3)
print(smoke[["image", "h", "w", "seconds", "max_prob", "top01_mean", "f1", "gt_pixels", "pred_pixels"]].round(3).to_string(index=False))
# Seconds per image (the first image includes warm-up, so use the last one as the typical speed)
typical = smoke["seconds"].iloc[-1]
print(f"\nTypical {typical:.2f} s/image -> about {typical * 300 / 60:.1f} min for all 300 images (P5)")

# Overlay figure: 3 pages (rows) x 4 panels (columns)
fig, axes = plt.subplots(3, 4, figsize=(16, 13))
for row, rec in enumerate(smoke.itertuples()):
    # Load the saved probability map (float16 -> float32), size h x w
    prob = np.load(f"{PROBS_DIR}/png/{rec.image}.npy").astype(np.float32)
    # The page shrunk to the same size, in grey (as the model sees it)
    page = Image.open(f"{INPUTS_DIR}/png/images/{rec.image}.png").convert("L").resize((rec.w, rec.h), Image.BILINEAR)
    # The box mask shrunk with nearest-neighbour (keeps it 0/1)
    box = Image.open(f"{INPUTS_DIR}/png/masks/{rec.image}.png").convert("L").resize((rec.w, rec.h), Image.NEAREST)
    # Panel 1: page
    axes[row, 0].imshow(page, cmap="gray")
    axes[row, 0].set_title(f"{rec.image} (model input, {rec.w}x{rec.h})")
    # Panel 2: answer key
    axes[row, 1].imshow(np.array(box) > 0, cmap="gray")
    axes[row, 1].set_title("Box mask (answer key)")
    # Panel 3: probability map with a colour bar from 0 to 1
    im = axes[row, 2].imshow(prob, cmap="magma", vmin=0, vmax=1)
    axes[row, 2].set_title(f"Tamper probability (max {rec.max_prob:.2f})")
    fig.colorbar(im, ax=axes[row, 2], fraction=0.046, label="probability")
    # Panel 4: thresholded map (official prediction)
    axes[row, 3].imshow(prob >= CONFIG["threshold"], cmap="gray")
    axes[row, 3].set_title(f"Prediction (prob >= 0.5), F1 = {rec.f1:.2f}")
    # Axis labels in model pixels
    for col in range(4):
        axes[row, col].set_xlabel("x (pixels at model size)")
        axes[row, col].set_ylabel("y (pixels)")
fig.suptitle("Smoke test: ADCD-Net on 3 forged bank-statement pages", y=0.995)
plt.tight_layout()
# Save the figure
plt.savefig(f"{RUN_DIR}/figures/smoke_overlay.png", dpi=110)
plt.show()

## 21. Sanity check, part 1: find and download the published test set (P4)

**WHY:** if our loop scores badly on our bank statements, there are two possible reasons: the model really struggles, *or* our setup is broken. To tell them apart we run the **same loop** on a test set the authors scored in their paper: **T-SROIE** (tampered receipts), from their Hugging Face dataset `Jason37437/Doc-Protocol-Data`, file `cutted_datasets_fakes.zip`. Like a control group in an experiment.

The cell first lists the files in that dataset with their sizes. **If the zip is over 2 GB it stops** (CLAUDE.md rule: ask first); tell Claude Code the size. Otherwise it downloads to `/tmp` (not saved as output) and unzips **only** the `T-SROIE_test` folder.

⏱ 1–5 min (download speed). ✅ A short file list with sizes, `downloaded ...`, and counts of `images`, `masks` (and maybe `ocr`) inside `T-SROIE_test`.

In [ ]:
# Tools to talk to Hugging Face (pre-installed on Kaggle)
from huggingface_hub import HfApi, hf_hub_download

# The published dataset and the file we need
SANITY_REPO = "Jason37437/Doc-Protocol-Data"
SANITY_FILE = "cutted_datasets_fakes.zip"
SANITY_SET = "T-SROIE_test"
# Download limit from CLAUDE.md
MAX_GB = 2.0
# Local folder for the download (in /tmp, so it is not saved as notebook output)
SANITY_DIR = "/tmp/sanity"
os.makedirs(SANITY_DIR, exist_ok=True)

# Ask Hugging Face for the list of files and their sizes
info = HfApi().dataset_info(SANITY_REPO, files_metadata=True)
files = pd.DataFrame([{"file": s.rfilename, "size_GB": round((s.size or 0) / 1e9, 3)} for s in info.siblings])
print(files.head(20).to_string(index=False))

# Size of the zip we want
zip_gb = files.loc[files["file"] == SANITY_FILE, "size_GB"].iloc[0]
print(f"\n{SANITY_FILE}: {zip_gb:.2f} GB")
# Stop if it is above the limit
if zip_gb > MAX_GB:
    raise SystemExit(f"STOP: {SANITY_FILE} is {zip_gb:.2f} GB (> {MAX_GB} GB). Tell Claude Code before downloading.")

# Download it (skipped automatically if already downloaded in this session)
zip_path = hf_hub_download(SANITY_REPO, SANITY_FILE, repo_type="dataset", local_dir=SANITY_DIR)
print("downloaded:", zip_path)

# Open the zip and keep only the files inside a T-SROIE_test folder
with zipfile.ZipFile(zip_path) as zf:
    members = [n for n in zf.namelist() if f"/{SANITY_SET}/" in "/" + n and not n.endswith("/")]
    # If nothing matched, show the top-level folders so we can find the right name
    if not members:
        tops = sorted({n.split("/")[0] for n in zf.namelist()})[:20]
        raise SystemExit(f"STOP: no {SANITY_SET} folder in the zip. Top-level names: {tops}")
    # Extract only those files
    zf.extractall(f"{SANITY_DIR}/extract", members=members)

# The T-SROIE_test folder on disk
SANITY_ROOT = os.path.join(f"{SANITY_DIR}/extract", members[0].split(f"{SANITY_SET}/")[0], SANITY_SET)
# Count the files in each sub-folder (images, masks, ocr, ...)
for sub in sorted(os.listdir(SANITY_ROOT)):
    sub_path = f"{SANITY_ROOT}/{sub}"
    if os.path.isdir(sub_path):
        sub_files = sorted(os.listdir(sub_path))
        print(f"{SANITY_SET}/{sub}: {len(sub_files)} files, e.g. {sub_files[:2]}")

## 22. Sanity check, part 2: OCR masks and path list (P4)

**WHY:** the loader needs an OCR text mask for every test image. If the published folder already has an `ocr/` folder, we use it and all images. If not, we make the masks with **the same OCR script** as for our pages. That costs time, so we then use a **random sample of `N_SANITY` = 100 images** (seed 42). The OCR script first times 3 images and the cell **stops if the estimate is over 30 minutes**.

Note: a sample's F1 will not exactly equal the paper's full-set F1. We give it a 95% confidence interval, so we can see whether the paper's value is inside it.

Then we build the path list `T-SROIE_sample.pkl` with the repo's rules (mask name = image name with `.jpg` → `.png`, OCR mask with the same name as the image).

⏱ under 1 min if OCR masks exist; otherwise ~3 s per receipt (estimate printed). ✅ `using the published OCR masks` *or* an OCR time estimate and `done`; then `T-SROIE_sample.pkl: N tuples, 0 missing`.

In [ ]:
# How many test images to use if we must make the OCR masks ourselves
CONFIG["N_SANITY"] = 100

# Image list of the published set, sorted
san_imgs = sorted(os.listdir(f"{SANITY_ROOT}/images"))
# Does the set come with OCR masks for every image?
have_ocr = os.path.isdir(f"{SANITY_ROOT}/ocr") and all(os.path.exists(f"{SANITY_ROOT}/ocr/{n}") for n in san_imgs)

if have_ocr:
    # Use every image and the published OCR masks
    sample = san_imgs
    san_ocr_dir = f"{SANITY_ROOT}/ocr"
    print(f"using the published OCR masks, all {len(sample)} images")
else:
    # Random sample of N_SANITY images, same seed every time
    sample = sorted(random.Random(CONFIG["seed"]).sample(san_imgs, min(CONFIG["N_SANITY"], len(san_imgs))))
    # Our own OCR masks go here
    san_ocr_dir = f"{SANITY_DIR}/ocr_ours"
    os.makedirs(san_ocr_dir, exist_ok=True)
    # Manifest for the OCR script: [label, image path, output path]
    san_manifest = [["sanity", f"{SANITY_ROOT}/images/{n}", f"{san_ocr_dir}/{n}"] for n in sample]
    san_manifest_path = f"{SANITY_DIR}/ocr_manifest_sanity.json"
    with open(san_manifest_path, "w") as f:
        json.dump(san_manifest, f)
    # Separate timing file for the sanity OCR
    san_timing = f"{RUN_DIR}/ocr_timing_sanity.csv"
    print(f"no published OCR masks: making our own for a sample of {len(sample)} images")
    # Time 3 images first
    run_stream([OCR_PY, f"{SCRIPTS_DIR}/ocr_masks.py", san_manifest_path, san_timing, "3"])
    # Estimate the rest
    sec = pd.read_csv(san_timing)["seconds"].median()
    left = sum(1 for _, _, out in san_manifest if not os.path.exists(out))
    print(f"median {sec:.1f} s/image, {left} left -> about {sec * left / 60:.0f} min")
    # Stop if it is over 30 minutes
    if sec * left / 60 > 30:
        raise SystemExit("STOP: sanity OCR estimate over 30 min. Tell Claude Code.")
    # Do the rest
    run_stream([OCR_PY, f"{SCRIPTS_DIR}/ocr_masks.py", san_manifest_path, san_timing, "0"])

# Build the path list with the repo's naming rules
san_list, n_missing = [], 0
for n in sample:
    # Image, mask (.jpg -> .png) and OCR mask (same name)
    img_p = f"{SANITY_ROOT}/images/{n}"
    mask_p = f"{SANITY_ROOT}/masks/{n.replace('.jpg', '.png')}"
    ocr_p = f"{san_ocr_dir}/{n}"
    # Keep the triple only if all three files exist
    if os.path.exists(img_p) and os.path.exists(mask_p) and os.path.exists(ocr_p):
        san_list.append((img_p, mask_p, ocr_p))
    else:
        n_missing += 1
# Save it next to our other path lists
with open(f"{PKL_DIR}/T-SROIE_sample.pkl", "wb") as f:
    pickle.dump(san_list, f)
print(f"T-SROIE_sample.pkl: {len(san_list)} tuples, {n_missing} missing")
# Record the choice in config.json
CONFIG["sanity"] = {"set": SANITY_SET, "n_images": len(san_list), "published_ocr": bool(have_ocr)}
with open(f"{RUN_DIR}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

## 23. Sanity check, part 3: run ADCD-Net and compare with the paper (P4)

**The metric: official-style F1.**
- **WHAT:** for each image, compare the predicted tampered pixels (probability ≥ 0.5) with the answer key, compute precision, recall and F1 *for that image*, then **average the F1 over all images**.
- **WHY:** it is the number the paper reports, so it is the only fair comparison. Averaging per image means every image counts equally, big or small edit.
- **HOW / formula:** for one image, $P = \frac{TP}{TP+FP}$, $R = \frac{TP}{TP+FN}$, $F1 = \frac{2PR}{P+R}$; then official F1 $= \frac{1}{N}\sum_i F1_i$.
- **Tiny example:** image A has F1 = 0.8, image B has F1 = 0.2 → official F1 = 0.5, even if B's edit is 10× bigger.

**Bootstrap 95% CI:** resample the images with replacement 1000 times, recompute the mean F1 each time, and take the 2.5th and 97.5th percentiles. Like re-drawing a sample from a population many times to see how much the average wobbles.

**Fill in `PAPER_F1`** with the paper's T-SROIE F1 for ADCD-Net (from the paper's table, as a number between 0 and 1). Leave it as `None` if you don't have it yet; the cell still runs.

**How to read it:** if the paper's value is inside or near our CI, the setup is right. If ours is **far lower**, the setup is wrong: stop and tell Claude Code before P5.

⏱ 1–3 min. ✅ `missing keys: 0 ...`, progress lines, and a small table: our F1 [CI] vs the paper's F1.

In [ ]:
# Reference F1 for ADCD-Net on T-SROIE_test (Doc Protocol slices). NOTE: the ADCD-Net paper itself does NOT
# report T-SROIE. 0.623 comes from two later papers (ForensicHub "Doc Protocol", second-hand), not from the authors.
PAPER_F1 = 0.623

# Run the same inference script on the published sample
run_stream(infer_cmd("T-SROIE_sample", "sanity_tsroie"), env=GPU_ENV)

# Read the per-image results
san = pd.read_csv(f"{SCORES_DIR}/sanity_tsroie.csv")

# Function: bootstrap 95% confidence interval of the mean of a column
def boot_ci(values, n_boot=CONFIG["N_BOOT"], seed=CONFIG["seed"]):
    # Random number generator with our seed
    rng = np.random.default_rng(seed)
    # The values as a numpy array
    v = np.asarray(values, dtype=float)
    # n_boot resamples of the same size, with replacement; mean of each
    means = rng.choice(v, size=(n_boot, len(v)), replace=True).mean(axis=1)
    # The 2.5th and 97.5th percentiles
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

# One row per metric: mean and its CI
rows = []
for metric in ["f1", "precision", "recall"]:
    lo, hi = boot_ci(san[metric])
    rows.append({"metric": f"official-style {metric}", "ours (mean)": round(san[metric].mean(), 3),
                 "95% CI": f"[{lo:.3f}, {hi:.3f}]"})
summary_san = pd.DataFrame(rows)
# Add the paper's F1 for comparison
summary_san["paper"] = [PAPER_F1 if PAPER_F1 is not None else "fill in PAPER_F1", "", ""]
print(f"Sanity check on {SANITY_SET} ({len(san)} images, published OCR masks: {CONFIG['sanity']['published_ocr']})")
print(summary_san.to_string(index=False))
# Is the reference value inside our 95% confidence interval?
if PAPER_F1 is not None:
    lo, hi = boot_ci(san["f1"])
    print(f"\nReference {PAPER_F1} is {'INSIDE' if lo <= PAPER_F1 <= hi else 'OUTSIDE'} our 95% CI [{lo:.3f}, {hi:.3f}]")
# Images whose answer key is empty (the Doc Protocol skips these; should be 0)
print("images with an empty answer key:", int((san["gt_pixels"] == 0).sum()))
print(f"\nSpeed: median {san['seconds'].median():.2f} s/image")
# Save the comparison table
summary_san.to_csv(f"{RUN_DIR}/sanity_check.csv", index=False)

# P5 — Full inference on all three variants

**WHY:** the smoke test and the sanity check showed the loop works, so now we run it on everything: `png` (100 tampered pages), `jpg90` (the same pages as JPEG q90) and `orig` (100 untouched originals). The same settings are used for all three, so any difference between them comes from the input, not the setup. Notebook 05 then only reads these maps; no GPU needed.

**Why not Save & Run All?** A fresh run would redo the 42-minute OCR. The full inference takes only a few minutes, so we run it **in this session**, zip the run folder and download it. Notebook 05 reads it as an uploaded Kaggle Dataset, the same way you uploaded `PATH_TO_TEXTCTRL_RUN_DATASET`.

## 24. Run all three variants, check, and zip (P5)

The cell:
1. runs `infer.py` on `png`, `jpg90`, `orig` (resumable: the 3 smoke-test pages are skipped);
2. merges the three score tables into `scores.csv` (one row per image and variant: max probability, 99.9th percentile, top-0.1 % mean, official-style P/R/F1) and `timing.csv` (image, variant, seconds);
3. runs the three checks: the same number of maps per variant, no failed image, and maps not all zero (min / max / mean of the score columns, **not** the maps themselves);
4. zips the run folder (without the page images, which are only links) to `/kaggle/working/<run name>.zip`.

For `orig`, the official F1 is always 0 because there is nothing to find. Originals are only used for false alarms and image-level AUC in notebook 05.

⏱ 2–4 min of inference + about 1 min to zip. ✅ A check table showing `100 / 100` maps for each variant and `failed: 0`; `max_prob` statistics that are not all 0; about 0.3 s/image; a zip of roughly 150–300 MB.

**After it runs:**
1. Download `/kaggle/working/<run name>.zip` (the exact name is printed).
2. On kaggle.com: **Create → New Dataset**, upload the zip, name it **`PATH_TO_ADCD_RUN_OUTPUTS_DATASET`**. Kaggle unzips it.
3. Then you can close this session.

In [ ]:
# Run the inference script on each variant (skips maps that already exist)
for variant in CONFIG["variants"]:
    print(f"\n===== {variant} =====")
    run_stream(infer_cmd(variant, variant), env=GPU_ENV)

# Merge the three per-variant tables into one, with a 'variant' column
frames = []
for variant in CONFIG["variants"]:
    # Read this variant's rows
    df = pd.read_csv(f"{SCORES_DIR}/{variant}.csv")
    # Add the variant name as the first column
    df.insert(0, "variant", variant)
    frames.append(df)
scores = pd.concat(frames, ignore_index=True)
# If an image was ever run twice, keep only the last row
scores = scores.drop_duplicates(subset=["variant", "image"], keep="last")
# Save the merged scores and the timing table
scores.to_csv(f"{RUN_DIR}/scores.csv", index=False)
scores[["image", "variant", "seconds"]].to_csv(f"{RUN_DIR}/timing.csv", index=False)

# Check 1 and 2: number of maps per variant vs number of images in its path list
check_rows = []
for variant in CONFIG["variants"]:
    # How many images the path list has
    with open(f"{PKL_DIR}/{variant}.pkl", "rb") as f:
        n_expected = len(pickle.load(f))
    # How many .npy maps exist (counted, never opened)
    n_maps = len([n for n in os.listdir(f"{PROBS_DIR}/{variant}") if n.endswith(".npy")])
    # How many rows the score table has
    n_rows = int((scores["variant"] == variant).sum())
    check_rows.append({"variant": variant, "maps": f"{n_maps} / {n_expected}", "score rows": n_rows,
                       "failed": n_expected - n_maps})
print("\n" + pd.DataFrame(check_rows).to_string(index=False))

# Check 3: the maps are not all zero (statistics of the score columns per variant)
print("\nScore columns per variant (min / max / mean):")
print(scores.groupby("variant")[["max_prob", "top01_mean", "mean_prob"]].agg(["min", "max", "mean"]).round(3))

# Timing: seconds per image and total
print("\nTiming per variant (seconds):")
print(scores.groupby("variant")["seconds"].agg(["median", "sum"]).round(2))
print(f"Total: {scores['seconds'].sum() / 60:.1f} min for {len(scores)} images")

# Zip the run folder, skipping the page-image links
RUN_ZIP = f"{WORK}/{RUN_NAME}.zip"
n_zipped = 0
with zipfile.ZipFile(RUN_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    # Walk through every folder of the run
    for folder, _, fnames in os.walk(RUN_DIR):
        for fname in fnames:
            path = os.path.join(folder, fname)
            # Skip links (the page images point to the read-only dataset)
            if os.path.islink(path):
                continue
            # Store it as <run name>/<path inside the run folder>
            zf.write(path, os.path.join(RUN_NAME, os.path.relpath(path, RUN_DIR)))
            n_zipped += 1
print(f"\nZipped {n_zipped} files -> {RUN_ZIP} ({os.path.getsize(RUN_ZIP) / 1e6:.0f} MB). Download it, then upload as the dataset 'PATH_TO_ADCD_RUN_OUTPUTS_DATASET'.")

In [ ]:
from IPython.display import FileLink
FileLink("adcd_20261026_1023.zip")